In [1]:
import torch
import torch.nn as nn

FEATURE_COLUMNS = [
    "creature_mass_talents", "oracle_omen", "aggression_index", "divine_aura",
    "aegis_resonance", "armor_resistance", "witness_count",
    "styx_disturbance", "omen_intensity"
]

class MythicThreatClassifier(nn.Module):
    """Tiny binary classifier shared by all three submitted checkpoints."""
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(len(FEATURE_COLUMNS), 1)

    def forward(self, x):
        return self.linear(x).squeeze(-1)

In [2]:
import pandas as pd

X = pd.read_csv('test_dataset.csv')
y = pd.read_csv('test_labels.csv')

test = pd.merge(X, y, on='encounter_id')

In [ ]:
# Just load everything
ARES = MythicThreatClassifier()
ATHENA = MythicThreatClassifier()
APOLLO = MythicThreatClassifier()

ARES.load_state_dict(torch.load('models/model_ares.pth', weights_only=True))
ATHENA.load_state_dict(torch.load('models/model_athena.pth', weights_only=True))
APOLLO.load_state_dict(torch.load('models/model_apollo.pth', weights_only=True))

<All keys matched successfully>

In [4]:
test_inputs = torch.tensor(test[FEATURE_COLUMNS].to_numpy(), dtype=torch.float32)

In [ ]:
import numpy as np
from sklearn.metrics import recall_score

# We don't want to miss a threat, so we value recall over every other metric
ARES_RECALL = recall_score(test['is_catastrophic_threat'], np.where(ARES(test_inputs) > 0, 1, 0))
ATHENA_RECALL = recall_score(test['is_catastrophic_threat'], np.where(ATHENA(test_inputs) > 0, 1, 0))
APOLLO_RECALL = recall_score(test['is_catastrophic_threat'], np.where(APOLLO(test_inputs) > 0, 1, 0))

print(f"Ares: {ARES_RECALL}, Athena: {ATHENA_RECALL}, Apollo: {APOLLO_RECALL}") # Obviously, we choose Ares here

Ares: 0.95, Athena: 0.75, Apollo: 0.55
